# Can search reject a wrong answer as reliably as it confirms a correct one?

Search agents often guess an answer and then search to verify it, so the query contains the guess. This pilot tests, on the development questions of BrowseComp-Plus, whether such a query retrieves documents that merely mention the guess instead of evidence that supports or refutes it.

Outputs of this notebook are committed, so they show only aggregates, `query_id` and `docid`, never text from the dataset.

## 0. Setup

Paths and the seed used throughout, and a check that the downloaded files are in place. Row counts and column names come from the parquet metadata, so no row is read here.

In [1]:
import base64
import hashlib
import json
import re
from pathlib import Path

import bm25s
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA = Path("data") if Path("data").exists() else Path("../data")
QUERIES_DIR = DATA / "browsecomp-plus" / "queries" / "data"
CORPUS_DIR = DATA / "browsecomp-plus" / "corpus" / "data"
RESULTS = DATA.parent / "results"
SEED = 0


def describe_parquet(directory: Path) -> None:
    files = sorted(directory.glob("*.parquet"))
    n_rows = sum(pq.ParquetFile(f).metadata.num_rows for f in files)
    columns = pq.ParquetFile(files[0]).schema_arrow.names
    print(f"{directory.parent.name}: {len(files)} files, {n_rows} rows, columns: {columns}")


describe_parquet(QUERIES_DIR)
describe_parquet(CORPUS_DIR)

queries: 6 files, 830 rows, columns: ['query_id', 'query', 'answer', 'gold_docs', 'negative_docs', 'evidence_docs']
corpus: 7 files, 100195 rows, columns: ['docid', 'text', 'url']


## 1. Development and test split

The questions are split at random before any result is computed, so that the choice of questions cannot depend on what we see. All prompts, query templates and thresholds are tuned on the development questions only; the test questions stay untouched until the final run. Only `query_id` is read here, and the split is saved to `results/split.json`.

In [2]:
N_DEV = 100
SPLIT_PATH = RESULTS / "split.json"

query_ids = sorted(
    query_id
    for f in QUERIES_DIR.glob("*.parquet")
    for query_id in pq.read_table(f, columns=["query_id"]).column("query_id").to_pylist()
)
shuffled = np.random.default_rng(SEED).permutation(query_ids).tolist()
split = {"seed": SEED, "dev": sorted(shuffled[:N_DEV]), "test": sorted(shuffled[N_DEV:])}

# The split must stay fixed once saved, so a mismatch stops the notebook instead of replacing it.
if SPLIT_PATH.exists():
    if json.loads(SPLIT_PATH.read_text()) != split:
        raise ValueError(f"{SPLIT_PATH} differs from the split computed here")
    print(f"{SPLIT_PATH} already exists and matches")
else:
    RESULTS.mkdir(exist_ok=True)
    SPLIT_PATH.write_text(json.dumps(split, indent=1) + "\n")
    print(f"Wrote {SPLIT_PATH}")

print(f"Questions: {len(query_ids)} total, {len(set(query_ids))} unique query_id")
print(f"Development: {len(split['dev'])}, test: {len(split['test'])}, "
      f"in both: {len(set(split['dev']) & set(split['test']))}")

../results/split.json already exists and matches
Questions: 830 total, 830 unique query_id
Development: 100, test: 730, in both: 0
